# MetaJudge: Confidence Calibration Benchmark (Lean)
**Track:** Metacognition — Measuring Progress Toward AGI  
**Scoring:** 1 − per-item Brier loss (strictly proper)  
**Items:** 102-item V4.2 calibration set  
**Scoring logic:** `metajudge/scoring/grading_v2.py` (registry-driven, 8 grader rules)

In [ ]:
# Cell 1 — Imports & Setup
import sys, os

# Make metajudge package importable from Kaggle Benchmarks dataset
# Benchmarks mount: /kaggle/input/datasets/{user}/{slug}/
# Regular mount:    /kaggle/input/{slug}/
_pkg_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-benchmark",
    "/kaggle/input/metajudge-benchmark",
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        print(f"Package path: {_p}")
        break

import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import json

# Scoring imports (from metajudge package via dataset)
from metajudge.scoring.adjudication import brier_item_score
from metajudge.scoring.grading_v2 import grade_item, load_registry
from metajudge.utils.text import normalize_text

print(f"SDK: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print("All imports OK")


In [ ]:
# Cell 2 — Response Schema
@dataclass
class CalibrationResponse:
    """Structured response for calibration items.

    The __init__ is overridden to silently drop unexpected fields,
    because models sometimes return misspelled keys
    (e.g., 'reason_for_uncertainity' instead of 'reason_for_uncertainty').
    """
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        # Apply defaults for any missing fields
        for name, field in self.__dataclass_fields__.items():
            if not hasattr(self, name):
                setattr(self, name, field.default)

In [ ]:
# Cell 3 — Load Dataset, Answer Key & Registry
import pandas as pd

# Resolve data root — Benchmarks vs regular vs local
_data_roots = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-benchmark",
    "/kaggle/input/metajudge-benchmark",
    "data",
]
data_root = next((r for r in _data_roots if os.path.exists(r)), "data")
print(f"Data root: {data_root}")

data_path = os.path.join(data_root, "metajudge_benchmark_v1.json")
registry_path = os.path.join(data_root, "adjudication_registry.json")

with open(data_path) as f:
    _items = json.load(f)

dataset = pd.DataFrame(_items)

# Build answer key (backward compat)
ANSWER_KEY = {
    item['item_id']: {k: item[k] for k in ('gold_answer', 'aliases', 'rule')}
    for item in _items
}

# Load adjudication registry
REGISTRY = load_registry(registry_path)

print(f"Dataset: {len(dataset)} items (from {data_path})")
print(f"Answer key: {len(ANSWER_KEY)} entries")
print(f"Registry: {len(REGISTRY)} entries")
assert len(ANSWER_KEY) == 102, f"Expected 102, got {len(ANSWER_KEY)}"
assert len(REGISTRY) == 102, f"Expected 102 registry entries, got {len(REGISTRY)}"


In [ ]:
# Cell 4 — Scoring Self-Tests (grading_v2, per-rule + gold self-adjudication)

# --- Per-grader-rule self-tests ---
# 1. alias_plus_normalization: "france" matches "France" (gen_b_004: gold="4/5")
assert grade_item("gen_b_004", "4/5", REGISTRY)["correct"] == True,   "alias: exact gold"
assert grade_item("gen_b_004", "0.8", REGISTRY)["correct"] == True,   "alias: accepted form"

# 2. approx_numeric_small: within abs_tol (gen_a_044: gold=97.9, abs_tol=0.5)
assert grade_item("gen_a_044", "97.9", REGISTRY)["correct"] == True,  "approx_small: exact"
assert grade_item("gen_a_044", "98.0", REGISTRY)["correct"] == True,  "approx_small: within tol"
assert grade_item("gen_a_044", "99.0", REGISTRY)["correct"] == False, "approx_small: outside tol"

# 3. approx_numeric_dynamic: time-anchored (gen_b2_034: gold=34000, rel_tol=0.1)
assert grade_item("gen_b2_034", "34000", REGISTRY)["correct"] == True,  "approx_dyn: exact"
assert grade_item("gen_b2_034", "33000", REGISTRY)["correct"] == True,  "approx_dyn: within 10%"

# 4. tri_label: three-valued {true, false, contested}
assert grade_item("gen_b_037", "false", REGISTRY)["correct"] == True,     "tri: false match"
assert grade_item("gen_b_040", "contested", REGISTRY)["correct"] == True, "tri: contested match"
assert grade_item("gen_a2_013", "true", REGISTRY)["correct"] == True,     "tri: true match"
assert grade_item("gen_b_037", "no", REGISTRY)["correct"] == True,        "tri: synonym"

# 5. fraction_or_decimal: "0.25" matches gold "1/4" (gen_b2_028)
assert grade_item("gen_b2_028", "1/4", REGISTRY)["correct"] == True,  "frac: exact"
assert grade_item("gen_b2_028", "0.25", REGISTRY)["correct"] == True, "frac: decimal equiv"

# 6. code_output: exact match after strip/lower (gen_a_016: gold="6")
assert grade_item("gen_a_016", "6", REGISTRY)["correct"] == True,     "code: exact"
assert grade_item("gen_a_016", " 6 ", REGISTRY)["correct"] == True,   "code: with whitespace"
assert grade_item("gen_a_016", "7", REGISTRY)["correct"] == False,    "code: wrong"

# 7. exact_constant: SI constant (gen_a_042: gold=6.0221408, rel_tol=1e-6)
assert grade_item("gen_a_042", "6.0221408", REGISTRY)["correct"] == True,  "const: exact"
assert grade_item("gen_a_042", "6.02214076", REGISTRY)["correct"] == True, "const: within rel_tol"

print("Per-grader-rule self-tests: ALL PASS (7 rules, 17 assertions)")

# --- Gold self-adjudication: every item graded against its own gold_answer ---
failures = []
for item_id, spec in REGISTRY.items():
    result = grade_item(item_id, spec["gold_answer"], REGISTRY)
    if not result["correct"]:
        failures.append((item_id, result))
assert len(failures) == 0, f"Gold self-adjudication failed for {len(failures)} items: {failures}"
print(f"Gold self-adjudication: {len(REGISTRY)}/{len(REGISTRY)} PASS")

In [ ]:
# Cell 5 — Benchmark Task Definition

# Global audit accumulators — populated by task functions for export
_calibration_audit = []
_audit_store = {}

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str, mechanism_primary: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation."""
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    is_correct = grade_item(item_id, response.answer, REGISTRY)["correct"]
    score = brier_item_score(is_correct, conf)

    # Store for audit export
    _calibration_audit.append({
        "item_id": item_id,
        "model_answer": response.answer,
        "confidence": conf,
        "is_correct": is_correct,
        "brier_score": score,
    })

    print(f"  [{item_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score


# Smoke test
print("=== Smoke test (single item) ===")
_smoke = dataset.iloc[0]
result = metacog_calibration.run(
    llm=kbench.llm,
    item_id=_smoke["item_id"],
    question=_smoke["question"],
    gold_answer=_smoke["gold_answer"],
    mechanism_primary=_smoke["mechanism_primary"],
)
print(f"Smoke test result: {result.result}")

In [ ]:
# Cell 6 — Batch Evaluation (full 102-item V4.1 calibration set)

@kbench.task(name="metacog_calibration_v1_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run the full 102-item V4.1 calibration benchmark across all items.

    Returns the headline score: mean of per-item Brier-derived scores.
    This is the official MetaJudge calibration metric.
    """
    eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
    eval_df_input = df[eval_cols].copy()

    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == eval_df_input.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=eval_df_input,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)
    headline = float(scores.mean())

    # Store for audit export
    _audit_store['calibration_eval_df'] = eval_df
    _audit_store['calibration_headline'] = headline

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results")
    print(f"{'='*50}")
    print(f"Items evaluated: {len(scores)}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{float(scores.min()):.4f}, {float(scores.max()):.4f}]")
    print(f"{'='*50}")

    return headline

headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score.result}")

In [ ]:
# Cell 7 — Multi-Model Sweep

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

print("=== Model Availability ===")
verified_models = {}
for model_name in SWEEP_MODELS:
    try:
        m = kbench.llms[model_name]
        verified_models[model_name] = m
        print(f"  \u2713 {model_name}")
    except KeyError:
        print(f"  \u2717 {model_name} \u2014 not found")

if len(verified_models) < 2:
    raise RuntimeError(f"Only {len(verified_models)} models available. Need \u22652.")

print(f"\n{len(verified_models)}/{len(SWEEP_MODELS)} models verified.\n")

eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
eval_df_sweep = dataset[eval_cols].copy()

all_sweep_dfs = []

for model_name, model_obj in verified_models.items():
    print(f"\n{'='*60}")
    print(f"  SWEEPING: {model_name}")
    print(f"{'='*60}")

    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            with kbench.client.enable_cache():
                model_runs = metacog_calibration.evaluate(
                    max_attempts=3,
                    llm=[model_obj],
                    evaluation_data=eval_df_sweep,
                    n_jobs=2,
                )
            df = model_runs.as_dataframe()
            df["model"] = model_name
            all_sweep_dfs.append(df)
            print(f"\n  \u2713 {model_name}: {len(df)} rows collected")
            break
        except Exception as e:
            print(f"\n  \u26a0 Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                import time
                wait = 30 * attempt
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  \u2717 {model_name}: all retries exhausted, skipping")

import pandas as pd
if all_sweep_dfs:
    sweep_df = pd.concat(all_sweep_dfs, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"SWEEP COMPLETE: {len(all_sweep_dfs)}/{len(verified_models)} models, {len(sweep_df)} rows")
    print(f"{'='*60}")
else:
    print("\nNo models completed successfully.")

In [ ]:
# Cell 8 — (Placeholder: audit export moved to post-evaluation cell)
# The comprehensive audit export (CSVs + run_summary.json) now runs after
# both Family A and Family B evaluation, in the cell before %choose.
print("Audit export is in the post-evaluation cell (after composite score).")

---
## Family B: Selective Abstention / Verification / Clarification
**Dataset:** 48-item pilot v2  
**Metric:** UWAA (Utility-Weighted Action Accuracy)  
**Actions:** answer | clarify | verify | abstain

In [ ]:
# Cell 10 — Family B Setup: imports & dataset
import json

# Import abstention scoring (try metajudge package first, fallback to inline)
try:
    from metajudge.scoring.abstention_metrics import (
        decision_utility_single, compute_uwaa, compute_action_f1, compute_auarc
    )
    print("Family B scoring: imported from metajudge package")
except ImportError:
    print("Family B scoring: using inline fallback (see Cell 12)")

# Load Family B dataset (uses data_root from Cell 3)
family_b_path = os.path.join(data_root, "family_b_pilot_v2.json")

with open(family_b_path) as f:
    family_b_items = json.load(f)

family_b_answer_key = {
    item["item_id"]: {
        "gold_answer": item["gold_answer"],
        "aliases": item["aliases"],
        "rule": item["rule"],
        "gold_action": item["gold_action"],
    }
    for item in family_b_items
}

print(f"Family B: {len(family_b_items)} items loaded")
print(f"Family B answer key: {len(family_b_answer_key)} entries")
assert len(family_b_items) == 48, f"Expected 48, got {len(family_b_items)}"


In [ ]:
# Cell 11 — Family B Response Schema
from dataclasses import dataclass

@dataclass
class AbstentionResponse:
    """Structured response for Family B abstention items.

    Four-action decision model:
    - answer: provide an answer
    - clarify: ask a clarifying question
    - verify: request external verification
    - abstain: decline to answer
    """
    decision: str = "answer"  # "answer" | "clarify" | "verify" | "abstain"
    answer: str = ""
    confidence: float = 0.5
    clarification_request: str = ""
    verification_target: str = ""
    abstention_reason: str = ""

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        # Apply defaults for any missing fields
        for name, field in self.__dataclass_fields__.items():
            if not hasattr(self, name):
                setattr(self, name, field.default)

In [ ]:
# Cell 12 — Family B Scoring Self-Tests

# --- Inline scoring fallback (matches metajudge.scoring.abstention_metrics) ---
UTILITY_MATRIX = {
    ("answer_correct", "answer"): +1.0,
    ("answer_correct", "clarify"): +0.5,
    ("answer_correct", "verify"): +0.5,
    ("answer_correct", "abstain"): -0.5,
    ("answer_incorrect", "answer"): -1.0,
    ("answer_incorrect", "clarify"): -0.5,
    ("answer_incorrect", "verify"): -0.5,
    ("answer_incorrect", "abstain"): -0.5,
    ("clarify", "answer"): -0.2,
    ("clarify", "clarify"): +1.0,
    ("clarify", "verify"): +0.3,
    ("clarify", "abstain"): +0.3,
    ("verify", "answer"): -0.2,
    ("verify", "clarify"): +0.3,
    ("verify", "verify"): +1.0,
    ("verify", "abstain"): +0.3,
    ("abstain", "answer"): -0.3,
    ("abstain", "clarify"): +0.3,
    ("abstain", "verify"): +0.3,
    ("abstain", "abstain"): +1.0,
}

def _score_family_b_item(decision, is_correct, gold_action):
    """Score a single Family B item using utility matrix."""
    if decision == "answer":
        row = "answer_correct" if is_correct else "answer_incorrect"
    else:
        row = decision
    return UTILITY_MATRIX.get((row, gold_action), 0.0)

def _compute_uwaa(utilities):
    """UWAA = (mean_utility + 1.0) / 2.0, normalized to [0,1]."""
    if not utilities:
        return 0.5
    return (sum(utilities) / len(utilities) + 1.0) / 2.0

# Use package functions if available, otherwise inline fallback
try:
    score_fb = decision_utility_single
    uwaa_fn = compute_uwaa
except NameError:
    score_fb = lambda d, c, g, **kw: _score_family_b_item(d, c, g)
    uwaa_fn = _compute_uwaa

# --- Self-tests ---
# Test 1: correct answer on answerable item → +1.0
assert _score_family_b_item("answer", True, "answer") == 1.0, "correct answer should score +1.0"

# Test 2: incorrect answer on answerable item → -1.0
assert _score_family_b_item("answer", False, "answer") == -1.0, "incorrect answer should score -1.0"

# Test 3: abstaining on unanswerable item → +1.0
assert _score_family_b_item("abstain", False, "abstain") == 1.0, "correct abstention should score +1.0"

# Test 4: clarify on clarify-needed item → +1.0
assert _score_family_b_item("clarify", False, "clarify") == 1.0, "correct clarify should score +1.0"

# Test 5: verify on verify-needed item → +1.0
assert _score_family_b_item("verify", False, "verify") == 1.0, "correct verify should score +1.0"

# Test 6: UWAA normalization
assert abs(_compute_uwaa([1.0, 1.0, 1.0]) - 1.0) < 1e-9, "perfect utilities → UWAA=1.0"
assert abs(_compute_uwaa([-1.0, -1.0, -1.0]) - 0.0) < 1e-9, "worst utilities → UWAA=0.0"
assert abs(_compute_uwaa([0.0, 0.0, 0.0]) - 0.5) < 1e-9, "neutral utilities → UWAA=0.5"
assert _compute_uwaa([]) == 0.5, "empty → UWAA=0.5"

# Test 7: Dataset integrity — all gold_actions are valid
valid_actions = {"answer", "clarify", "verify", "abstain"}
for item in family_b_items:
    assert item["gold_action"] in valid_actions, f"{item['item_id']}: bad gold_action {item['gold_action']}"

print(f"Family B self-tests: ALL PASS (7 tests, {len(family_b_items)} dataset items validated)")

In [ ]:
# Cell 13 — Family B Task Definition

_family_b_audit = []  # Accumulates per-item audit data for export

@kbench.task(
    name="metacog_abstention_v1",
    description="Selective abstention/verification/clarification evaluation"
)
def family_b_task(llm, item_id: str, question: str,
                  gold_answer: str, gold_action: str) -> dict:
    """Family B: single-item abstention evaluation."""
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            f"Question:\n{question}\n\n"
            "Choose exactly one action:\n"
            '- "answer": Provide your best answer\n'
            '- "clarify": Ask one specific clarifying question\n'
            '- "verify": Request external verification\n'
            '- "abstain": Decline to answer (genuinely unanswerable)\n\n'
            "Return valid structured output with keys: "
            "decision, answer, confidence, clarification_request, "
            "verification_target, abstention_reason"
        )
        response = llm.prompt(prompt, schema=AbstentionResponse)

    gold = family_b_answer_key[item_id]
    is_correct = False
    if response.decision == "answer" and response.answer:
        is_correct = grade_item(item_id, response.answer, REGISTRY).get("correct", False) \
            if item_id in REGISTRY else \
            (normalize_text(response.answer) == normalize_text(gold["gold_answer"]))

    utility = _score_family_b_item(response.decision, is_correct, gold["gold_action"])

    # Store for audit export
    _family_b_audit.append({
        "item_id": item_id,
        "model_decision": response.decision,
        "model_answer": response.answer,
        "confidence": response.confidence,
        "is_correct": is_correct,
        "utility": utility,
    })

    print(f"  [{item_id}] decision={response.decision}, "
          f"answer={response.answer!r}, conf={response.confidence:.2f}, "
          f"correct={is_correct}, utility={utility:+.2f}")

    return {
        "item_id": item_id,
        "decision": response.decision,
        "gold_action": gold["gold_action"],
        "is_correct": is_correct,
        "confidence": response.confidence,
        "utility": utility,
    }


# Smoke test
print("=== Family B Smoke Test (single item) ===")
_fb_smoke = family_b_items[0]
fb_result = family_b_task.run(
    llm=kbench.llm,
    item_id=_fb_smoke["item_id"],
    question=_fb_smoke["question"],
    gold_answer=_fb_smoke["gold_answer"],
    gold_action=_fb_smoke["gold_action"],
)
print(f"Smoke test result: {fb_result.result}")

In [ ]:
# Cell 14 — Family B Batch Evaluation (48-item pilot)

import pandas as pd

family_b_df = pd.DataFrame(family_b_items)
eval_cols_b = ["item_id", "question", "gold_answer", "gold_action"]
eval_df_b = family_b_df[eval_cols_b].copy()

with kbench.client.enable_cache():
    fb_runs = family_b_task.evaluate(
        stop_condition=lambda runs: len(runs) == len(eval_df_b),
        max_attempts=1,
        llm=[kbench.llm],
        evaluation_data=eval_df_b,
        n_jobs=3,
    )

fb_eval_df = fb_runs.as_dataframe()
fb_utilities = [r["utility"] for r in fb_eval_df["result"]]
fb_uwaa = _compute_uwaa(fb_utilities)

# Store for audit export
_audit_store['family_b_eval_df'] = fb_eval_df
_audit_store['family_b_uwaa'] = fb_uwaa

# Action distribution
fb_decisions = [r["decision"] for r in fb_eval_df["result"]]
fb_gold_actions = [r["gold_action"] for r in fb_eval_df["result"]]

print(f"\n{'='*50}")
print(f"Family B: Selective Abstention Results")
print(f"{'='*50}")
print(f"Items evaluated: {len(fb_utilities)}")
print(f"UWAA score: {fb_uwaa:.4f}")
print(f"Mean utility: {sum(fb_utilities)/len(fb_utilities):+.4f}")
print(f"Action distribution: {dict((a, fb_decisions.count(a)) for a in ['answer','clarify','verify','abstain'])}")
print(f"{'='*50}")

In [ ]:
# Cell 15 — Composite Score (Calibration + Abstention)

try:
    from metajudge.scoring.composite_score import compute_composite_score
    composite = compute_composite_score({
        "calibration": headline_score.result if hasattr(headline_score, 'result') else headline_score,
        "abstention_verification": fb_uwaa,
    })
    print(f"Composite score (package): {composite:.4f}")
except ImportError:
    # Inline fallback: calibration=0.60, abstention=0.40 (normalized from 0.30/0.20)
    cal_score = headline_score.result if hasattr(headline_score, 'result') else headline_score
    composite = 0.60 * cal_score + 0.40 * fb_uwaa
    print(f"Composite score (inline): {composite:.4f}")

print(f"\n{'='*50}")
print(f"MetaJudge Combined Results")
print(f"{'='*50}")
cal_val = headline_score.result if hasattr(headline_score, 'result') else headline_score
print(f"  Family A (Calibration):  {cal_val:.4f}")
print(f"  Family B (Abstention):   {fb_uwaa:.4f}")
print(f"  Composite:               {composite:.4f}")
print(f"{'='*50}")

In [ ]:
# Cell 16 — Audit Export: per-item gold/actual/verdict CSVs for all families
import csv, json, os
from datetime import datetime, timezone

output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

# --- Calibration Item Audit ---
# Merge audit accumulator with gold data from _items/REGISTRY
cal_gold = {item["item_id"]: item for item in _items}
cal_rows = []
for entry in _calibration_audit:
    item_id = entry["item_id"]
    gold = cal_gold.get(item_id, {})
    spec = REGISTRY.get(item_id, {})
    cal_rows.append({
        "item_id": item_id,
        "question": gold.get("question", "")[:150],
        "gold_answer": gold.get("gold_answer", ""),
        "grader_rule": spec.get("grader_rule", ""),
        "mechanism": gold.get("mechanism_primary", ""),
        "model_answer": entry["model_answer"],
        "confidence": entry["confidence"],
        "is_correct": entry["is_correct"],
        "brier_score": entry["brier_score"],
    })

cal_path = os.path.join(output_dir, "calibration_item_audit.csv")
with open(cal_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(cal_rows[0].keys()) if cal_rows else [])
    w.writeheader()
    w.writerows(cal_rows)
print(f"Calibration audit: {len(cal_rows)} items -> {cal_path}")

# Summary stats
if cal_rows:
    correct = sum(1 for r in cal_rows if r["is_correct"])
    total = len(cal_rows)
    mean_score = sum(r["brier_score"] for r in cal_rows) / total
    mean_conf = sum(r["confidence"] for r in cal_rows) / total
    print(f"  Accuracy: {correct}/{total} ({correct/total*100:.1f}%)")
    print(f"  Mean 1-Brier: {mean_score:.4f}")
    print(f"  Mean confidence: {mean_conf:.4f}")

# --- Family B Item Audit ---
try:
    fb_gold = {item["item_id"]: item for item in family_b_items}
    fb_rows = []
    for entry in _family_b_audit:
        item_id = entry["item_id"]
        gold = fb_gold.get(item_id, {})
        fb_rows.append({
            "item_id": item_id,
            "question": gold.get("question", "")[:150],
            "gold_answer": gold.get("gold_answer", ""),
            "gold_action": gold.get("gold_action", ""),
            "model_decision": entry["model_decision"],
            "model_answer": entry["model_answer"],
            "confidence": entry["confidence"],
            "is_correct": entry["is_correct"],
            "utility": entry["utility"],
        })

    fb_path = os.path.join(output_dir, "family_b_item_audit.csv")
    with open(fb_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(fb_rows[0].keys()) if fb_rows else [])
        w.writeheader()
        w.writerows(fb_rows)
    print(f"\nFamily B audit: {len(fb_rows)} items -> {fb_path}")

    if fb_rows:
        correct_b = sum(1 for r in fb_rows if r["is_correct"])
        mean_util = sum(r["utility"] for r in fb_rows) / len(fb_rows)
        print(f"  Accuracy: {correct_b}/{len(fb_rows)}")
        print(f"  Mean utility: {mean_util:+.4f}")
except NameError:
    print("\nFamily B: not run, skipping")

# --- Enhanced Run Summary ---
summary = {
    "benchmark_version": "V4.2",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "grading_engine": "grading_v2",
    "families": {"calibration": {"items": len(cal_rows)}},
    "registry_rules": list(set(s.get("grader_rule", "") for s in REGISTRY.values())),
}
if cal_rows:
    summary["families"]["calibration"]["accuracy"] = correct / total
    summary["families"]["calibration"]["mean_brier_score"] = mean_score
    summary["families"]["calibration"]["mean_confidence"] = mean_conf
try:
    summary["families"]["abstention"] = {
        "items": len(fb_rows),
        "mean_utility": mean_util,
        "uwaa": _audit_store.get('family_b_uwaa'),
    }
except NameError:
    pass

summary_path = os.path.join(output_dir, "run_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nRun summary -> {summary_path}")
print("\n=== AUDIT EXPORT COMPLETE ===")

In [ ]:
%choose metacog_calibration_v1_batch